<a href="https://colab.research.google.com/github/praveenvadde1250/fittrack-project-2601018/blob/main/EmployeeAnalysis_Vadde_Praveen_2601018.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NAME: Vadde Praveen

ID: 2601018

In [17]:
#=============================================================#
# Task 1: Create Database#
# Import required libraries
import sqlite3
import random
import pandas as pd
import os # Import the os module for file operations

# Define database file name
DB_FILE = 'employees.db'

# Check if the database file exists and remove it to ensure a fresh start.
# This is a robust way to handle 'database is locked' errors in Colab/Jupyter notebooks
# where previous connections might not be properly closed by the kernel.
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)

# Create database (this will create the file if it doesn't exist)
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Create table
cursor.execute('''
CREATE TABLE IF NOT EXISTS employees (
    emp_id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary REAL,
    years_experience INTEGER,
    performance_score REAL
)
''')

# Departments list
departments = ['Engineering', 'Sales', 'Marketing', 'HR', 'Finance']

# Insert 40 records
for i in range(1, 41):
    name = f'Employee_{i}'
    department = random.choice(departments)
    salary = round(random.uniform(50000, 150000), 2)
    years_experience = random.randint(1, 15)
    performance_score = round(random.uniform(1.0, 5.0), 2)

    cursor.execute('''
    INSERT INTO employees (emp_id, name, department, salary, years_experience, performance_score)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', (i, name, department, salary, years_experience, performance_score))

conn.commit()
conn.close() # Ensure the connection is closed after all operations

In [8]:
# Task 2: SQL Queries (Load into Pandas)

query1 = '''
SELECT name, department, salary, performance_score
FROM employees
WHERE performance_score >= 4.0 AND years_experience >= 3
ORDER BY performance_score DESC
LIMIT 15;
'''

df_q1 = pd.read_sql(query1, conn)
df_q1

,name,department,salary,performance_score
0,Employee_32,Marketing,148434.82,5.00
1,Employee_12,Sales,140758.21,4.96
2,Employee_4,Marketing,118857.68,4.82
3,Employee_29,Finance,120207.03,4.73
4,Employee_14,Marketing,78942.87,4.63
5,Employee_15,HR,109121.42,4.50
6,Employee_2,Finance,59123.90,4.46
7,Employee_8,Sales,59503.03,4.43
8,Employee_19,HR,67757.40,4.26
9,Employee_38,Engineering,137711.90,4.23


In [9]:
#Query 2

query2 = '''
SELECT *
FROM employees
WHERE salary BETWEEN 70000 AND 110000
AND department IN ('Engineering', 'Sales')
ORDER BY department ASC, salary DESC;
'''

df_q2 = pd.read_sql(query2, conn)
df_q2

,emp_id,name,department,salary,years_experience,performance_score
0,36,Employee_36,Engineering,105142.34,6,2.96
1,1,Employee_1,Sales,106464.41,4,1.18
2,37,Employee_37,Sales,100017.54,8,2.98
3,35,Employee_35,Sales,82457.77,4,1.39
4,20,Employee_20,Sales,80763.05,11,2.73


In [10]:
#Query 3

query3 = '''
SELECT department, COUNT(*) AS employee_count, AVG(salary) AS avg_salary
FROM employees
GROUP BY department
ORDER BY avg_salary DESC;
'''

df_q3 = pd.read_sql(query3, conn)
df_q3

,department,employee_count,avg_salary
0,Marketing,9,100904.465556
1,Finance,7,93366.091429
2,HR,9,92103.772222
3,Sales,9,91318.808889
4,Engineering,6,87564.866667


Task 3: Pandas Analysis

Average Performance Score by Department

In [11]:
df_all = pd.read_sql("SELECT * FROM employees", conn)

avg_perf = df_all.groupby('department')['performance_score'].mean().reset_index()
avg_perf

,department,performance_score
0,Engineering,2.660000
1,Finance,3.240000
2,HR,2.732222
3,Marketing,2.957778
4,Sales,3.161111


In [12]:
# Replicate Query 1 using ONLY Pandas
df_pandas_q1 = (
    df_all.loc[
        (df_all['performance_score'] >= 4.0) &
        (df_all['years_experience'] >= 3)
    ]
    .sort_values(by='performance_score', ascending=False)
    .head(15)
)

df_pandas_q1

,emp_id,name,department,salary,years_experience,performance_score
31,32,Employee_32,Marketing,148434.82,4,5.00
11,12,Employee_12,Sales,140758.21,11,4.96
3,4,Employee_4,Marketing,118857.68,13,4.82
28,29,Employee_29,Finance,120207.03,4,4.73
13,14,Employee_14,Marketing,78942.87,10,4.63
14,15,Employee_15,HR,109121.42,9,4.50
1,2,Employee_2,Finance,59123.90,7,4.46
7,8,Employee_8,Sales,59503.03,8,4.43
18,19,Employee_19,HR,67757.40,5,4.26
37,38,Employee_38,Engineering,137711.90,15,4.23


Task 4: SQL vs Pandas (150–200 words)

SQL and Pandas are both powerful tools for data analysis, but they serve different purposes and are used in different contexts. SQL uses declarative syntax such as SELECT, WHERE, AND, and ORDER BY to query structured data stored in databases. In contrast, Pandas uses Python-based syntax like ==, &, .loc[], and .sort_values() to manipulate in-memory DataFrames.

SQL is best suited for working with large datasets that reside in databases, especially when the data cannot fit into memory. It allows efficient filtering, aggregation, and joining directly within the database engine. Pandas, on the other hand, is ideal for exploratory data analysis, data cleaning, and transformations when the dataset fits into memory.

A key difference is that SQL operations are executed on the database side, while Pandas operations are performed in-memory, making Pandas more flexible but less scalable for very large datasets. Therefore, SQL is preferred for large-scale data processing, while Pandas is better for detailed analysis, visualization, and machine learning workflows.